In [1]:
import os
import subprocess
os.chdir("..")
print(os.getcwd())  # 打印當前工作目錄的絕對路徑

d:\pythonProject\IC Lab\Gait_analysis\pyskl


In [2]:
from colab.utils.datasets_npz_aug_2c_flip import load_npz_dataset
# from colab.utils.datasets_npz_ori import load_npz_dataset
import torch

def get_dataset_profile(split=5):
    print(f"split : {split}")
    # 設定變數
    num_point = 17
    input_channel = 2
    pose = "yolo"  # 需要替換為實際值
    view = "sagittal"  # 需要替換為實際值
    frame = 90  # 需要替換為實際 frame 數值
    cls = 4  # 需要替換為實際分類數
    batch_size = 1  # 需要替換為實際 batch size
    num_workers = 4  # 需要替換為實際 workers 數
    feature = "j"  # 需要替換為實際特徵選擇

    # 定義檔案路徑
    train_file = f"data/drunk71_small/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_train.npz"
    validation_file = f"data/drunk71_small/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_val.npz"
    test_file = f"data/drunk71_small/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_test.npz"

    # 載入數據集
    training_loader = load_npz_dataset(train_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)
    validation_loader = load_npz_dataset(validation_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)
    test_loader = load_npz_dataset(test_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)

    
    # **統計 Training / Validation / Test 數據集的類別數量**
    for name, loader in zip(["Training", "Validation", "Test"], [training_loader, validation_loader, test_loader]):  
        counts = class_count(loader, 4)  # 這裡 counts 現在有值
        print(f"\n{name} set class distribution:")
        for i, count in enumerate(counts):  # **修正 cls → i**
            print(f"Class {i}: {int(count)}")  # **確保轉換為 int 避免 PyTorch tensor 格式輸出**

def class_count(loader, num_classes):
    class_counts = torch.zeros(num_classes, dtype=torch.float32)
    for _, batch in enumerate(loader):  
        labels = batch['label'].view(-1)  # 確保 labels 是一維
        for c in range(num_classes):
            class_counts[c] += (labels == c).sum().item()
    
    return class_counts  # **新增 return，回傳統計結果**

In [ ]:
get_dataset_profile(split = 8)
get_dataset_profile(split = 9)
# train 4512
# val 1128
# test 1440

split : 8

Training set class distribution:
Class 0: 2256
Class 1: 752
Class 2: 752
Class 3: 752

Validation set class distribution:
Class 0: 564
Class 1: 188
Class 2: 188
Class 3: 188

Test set class distribution:
Class 0: 720
Class 1: 240
Class 2: 240
Class 3: 240
split : 9

Training set class distribution:
Class 0: 2256
Class 1: 752
Class 2: 752
Class 3: 752

Validation set class distribution:
Class 0: 564
Class 1: 188
Class 2: 188
Class 3: 188

Test set class distribution:
Class 0: 720
Class 1: 240
Class 2: 240
Class 3: 240


: 

In [3]:
# from colab.utils.datasets_npz_aug_2c_flip import load_npz_dataset
from colab.utils.datasets_npz_ori import load_npz_dataset
import torch

split = 5 # 需要替換為實際 split 數值

# 設定變數
num_point = 17
input_channel = 2
pose = "yolo"  # 需要替換為實際值
view = "sagittal"  # 需要替換為實際值
frame = 90  # 需要替換為實際 frame 數值
cls = 4  # 需要替換為實際分類數
batch_size = 1  # 需要替換為實際 batch size
num_workers = 4  # 需要替換為實際 workers 數
feature = "j"  # 需要替換為實際特徵選擇

# 定義檔案路徑
train_file = f"data/drunk71/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_train.npz"
validation_file = f"data/drunk71/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_val.npz"
test_file = f"data/drunk71/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_test.npz"

# 載入數據集
training_loader = load_npz_dataset(train_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)
validation_loader = load_npz_dataset(validation_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)
test_loader = load_npz_dataset(test_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)


In [4]:
def print_names_and_labels(loader, set_name):
    print(f"\n🗂️ {set_name} Loader Contents:")
    for batch_idx, batch in enumerate(loader):
        labels = batch['label'].view(-1).tolist()

        # 假設 'name' 是 list of str，或是一個 list-like 的資料
        if isinstance(batch['name'], (list, tuple)):
            names = batch['name']
        elif isinstance(batch['name'], torch.Tensor):
            names = batch['name'].view(-1).tolist()
        else:
            print(f"⚠️ Unknown type for 'name': {type(batch['name'])}")
            names = ["N/A"] * len(labels)

        for name, label in zip(names, labels):
            print(f" - {name}: Class {label}")

def collect_subject_ids(loader):
    subject_ids = set()
    for batch in loader:
        if isinstance(batch['name'], (list, tuple)):
            names = batch['name']
        elif isinstance(batch['name'], torch.Tensor):
            names = batch['name'].view(-1).tolist()
        else:
            continue

        for name in names:
            sid = name[:2]  # 取前兩碼作為 subject id
            subject_ids.add(sid)
    return subject_ids


In [ ]:
# for name, loader in zip(["Training", "Validation", "Test"], [training_loader, validation_loader, test_loader]):
#     print_names_and_labels(loader, name)

# # 然後做分析
# train_ids = collect_subject_ids(training_loader)
# val_ids = collect_subject_ids(validation_loader)
# test_ids = collect_subject_ids(test_loader)

# # 印 ID & 重複
# print("👟 Training Subject IDs:", sorted(train_ids))
# print("🧪 Validation Subject IDs:", sorted(val_ids))
# print("📦 Test Subject IDs:", sorted(test_ids))

# # 檢查重複與缺失
# print("\n🔁 Overlapping Subjects:")
# print("Train & Val:", sorted(train_ids & val_ids))
# print("Train & Test:", sorted(train_ids & test_ids))
# print("Val & Test:", sorted(val_ids & test_ids))

# all_ids_used = train_ids | val_ids | test_ids
# expected_ids = set(f"{i:02}" for i in range(1, 72))  # 01 ~ 71
# print(f"\n🧾 Total unique subjects used: {len(all_ids_used)}")
# print("📋 All used:", sorted(all_ids_used))
# missing = expected_ids - all_ids_used
# if missing:
#     print("❌ Missing subject IDs:", sorted(missing))
# else:
#     print("✅ All subject IDs (01~71) are used.")


🗂️ Training Loader Contents:
 - 26_sagittal_medium_04: Class 2
 - 38_sagittal_normal_10: Class 0
 - 27_sagittal_high_10: Class 3
 - 33_sagittal_low_01: Class 1
 - 26_sagittal_low_06: Class 1
 - 10_sagittal_low_03: Class 1
 - 46_sagittal_low_08: Class 1
 - 39_sagittal_medium_10: Class 2
 - 03_sagittal_normal_08: Class 0
 - 09_sagittal_low_04: Class 1
 - 60_sagittal_low_03: Class 1
 - 36_sagittal_medium_04: Class 2
 - 37_sagittal_high_08: Class 3
 - 71_sagittal_normal_09: Class 0
 - 57_sagittal_normal_10: Class 0
 - 60_sagittal_medium_06: Class 2
 - 34_sagittal_normal_07: Class 0
 - 01_sagittal_normal_10: Class 0
 - 25_sagittal_low_05: Class 1
 - 48_sagittal_medium_08: Class 2
 - 07_sagittal_high_10: Class 3
 - 37_sagittal_normal_05: Class 0
 - 17_sagittal_high_01: Class 3
 - 48_sagittal_high_09: Class 3
 - 70_sagittal_normal_10: Class 0
 - 57_sagittal_normal_06: Class 0
 - 25_sagittal_normal_01: Class 0
 - 28_sagittal_medium_03: Class 2
 - 58_sagittal_normal_04: Class 0
 - 46_sagittal_

# 使用split 5 重新劃分 split 6 7 

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import ConcatDataset

split = 22

# Step 1: 讀 CSV 並建立 split 對應字典
df = pd.read_csv(r'D:\pythonProject\IC Lab\Gait_analysis\pyskl\tools\data\drunk\crop_subject_kfold_test.csv')
df['subject_id'] = df['subject'].apply(lambda x: f"{int(x):02}")
split_dict = dict(zip(df['subject_id'], df[f'split_{split}']))

# 🔍 印出這個 split 中的 train/test subject IDs
train_ids = sorted([sid for sid, s in split_dict.items() if s == 'train'])
test_ids = sorted([sid for sid, s in split_dict.items() if s == 'test'])

print(f"🟩 Split {split} - Train Subjects ({len(train_ids)}): {train_ids}")
print(f"🟥 Split {split} - Test Subjects ({len(test_ids)}): {test_ids}")

# Step 2: 合併三個 DataLoader 成一個完整資料集
all_dataset = ConcatDataset([training_loader.dataset, validation_loader.dataset, test_loader.dataset])
all_names, all_labels, all_keypoints = [], [], []

for item in all_dataset:
    all_names.append(item['name'])
    all_labels.append(item['label'])
    all_keypoints.append(item['keypoint'])

all_names = np.array(all_names)
all_labels = np.array(all_labels)
all_keypoints = np.array(all_keypoints)

# Step 3: 根據前兩碼取得 subject id 並查 split
subject_ids = np.array([name[:2] for name in all_names])
split_tags = np.array([split_dict.get(sid, 'unknown') for sid in subject_ids])

# Step 4: 先過濾掉 val subject 的資料
mask_not_val = split_tags != 'val'
names_kept = all_names[mask_not_val]
labels_kept = all_labels[mask_not_val]
keypoints_kept = all_keypoints[mask_not_val]
subject_ids_kept = subject_ids[mask_not_val]
split_tags_kept = split_tags[mask_not_val]

# Step 5: 分出 test / train
mask_test = split_tags_kept == 'test'
print(mask_test)
mask_train = split_tags_kept == 'train'
print(mask_train)

# Test set
test_names = names_kept[mask_test]
test_labels = labels_kept[mask_test]
test_keypoints = keypoints_kept[mask_test]
np.savez(f"data/drunk71/dataset_yolo_sagittal_split{split}_90frames_j_4class_test.npz",
         keypoints=test_keypoints, labels=test_labels, names=test_names)

# Train set → 再 stratified 拆成 train/val
train_names = names_kept[mask_train]
train_labels = labels_kept[mask_train]
train_keypoints = keypoints_kept[mask_train]

kps_train, kps_val, lbls_train, lbls_val, names_train, names_val = train_test_split(
    train_keypoints, train_labels, train_names,
    test_size=0.2,
    stratify=train_labels,
    random_state=1234
)

np.savez(f"data/drunk71/dataset_yolo_sagittal_split{split}_90frames_j_4class_train.npz",
         keypoints=kps_train, labels=lbls_train, names=names_train)

np.savez(f"data/drunk71/dataset_yolo_sagittal_split{split}_90frames_j_4class_val.npz",
         keypoints=kps_val, labels=lbls_val, names=names_val)


🟩 Split 22 - Train Subjects (16): ['07', '08', '09', '14', '33', '34', '41', '58', '61', '63', '64', '65', '67', '68', '69', '70']
🟥 Split 22 - Test Subjects (4): ['25', '59', '66', '71']
[ True  True False False False False False False False False False False
 False False  True False False  True False False False False False  True
 False False False False False False False False False False False False
 False False False False False False False False False  True  True False
 False False  True  True  True False False False False False False False
 False False False  True False False False False False False False False
 False False  True False False False  True False  True False False False
  True False False False  True False False False False False  True False
  True False False False  True  True False  True  True False  True False
 False False False  True  True False False False False False False  True
 False False  True False  True  True False  True False  True  True False
 False Fa

In [ ]:
from colab.utils.datasets_npz_aug_2c_flip import load_npz_dataset
import torch

def get_dataset_profile(split=5):
    print(f"split : {split}")
    # 設定變數
    num_point = 17
    input_channel = 2
    pose = "yolo"  # 需要替換為實際值
    view = "sagittal"  # 需要替換為實際值
    frame = 90  # 需要替換為實際 frame 數值
    cls = 4  # 需要替換為實際分類數
    batch_size = 1  # 需要替換為實際 batch size
    num_workers = 4  # 需要替換為實際 workers 數
    feature = "j"  # 需要替換為實際特徵選擇

    # 定義檔案路徑
    train_file = f"data/drunk71_small/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_train.npz"
    validation_file = f"data/drunk71_small/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_val.npz"
    test_file = f"data/drunk71_small/dataset_{pose}_{view}_split{split}_{frame}frames_j_{cls}class_test.npz"

    # 載入數據集
    training_loader = load_npz_dataset(train_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)
    validation_loader = load_npz_dataset(validation_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)
    test_loader = load_npz_dataset(test_file, workers_per_gpu=num_workers, batch_size=batch_size, pose=pose, feature=feature)

    
    # **統計 Training / Validation / Test 數據集的類別數量**
    for name, loader in zip(["Training", "Validation", "Test"], [training_loader, validation_loader, test_loader]):  
        counts = class_count(loader, 4)  # 這裡 counts 現在有值
        # print(f"\n{name} set class distribution:")
        # for i, count in enumerate(counts):  # **修正 cls → i**
        #     print(f"Class {i}: {int(count)}")  # **確保轉換為 int 避免 PyTorch tensor 格式輸出**
        
        # 額外統計 Test set 的人員 ID
        if name == "Test":
            subject_ids = set()
            for _, batch in enumerate(loader):
                names = batch['name']
                for n in names:
                    subject_id = str(n[:2]) if isinstance(n, str) else str(n.numpy()[0:2].tobytes().decode())
                    subject_ids.add(subject_id)
            print(f"\nTest set subject IDs: {sorted(subject_ids)}")

def class_count(loader, num_classes):
    class_counts = torch.zeros(num_classes, dtype=torch.float32)
    for _, batch in enumerate(loader):  
        labels = batch['label'].view(-1)  # 確保 labels 是一維
        for c in range(num_classes):
            class_counts[c] += (labels == c).sum().item()
    
    return class_counts  # **新增 return，回傳統計結果**

In [ ]:
get_dataset_profile(8)
get_dataset_profile(9)
get_dataset_profile(12)
get_dataset_profile(13)
get_dataset_profile(14)
get_dataset_profile(15)

split : 12

Training set class distribution:
Class 0: 2256
Class 1: 752
Class 2: 752
Class 3: 752

Validation set class distribution:
Class 0: 564
Class 1: 188
Class 2: 188
Class 3: 188

Test set class distribution:
Class 0: 720
Class 1: 240
Class 2: 240
Class 3: 240
split : 13

Training set class distribution:
Class 0: 2256
Class 1: 752
Class 2: 752
Class 3: 752

Validation set class distribution:
Class 0: 564
Class 1: 188
Class 2: 188
Class 3: 188

Test set class distribution:
Class 0: 720
Class 1: 240
Class 2: 240
Class 3: 240
split : 14

Training set class distribution:
Class 0: 2256
Class 1: 752
Class 2: 752
Class 3: 752

Validation set class distribution:
Class 0: 564
Class 1: 188
Class 2: 188
Class 3: 188

Test set class distribution:
Class 0: 720
Class 1: 240
Class 2: 240
Class 3: 240
split : 15

Training set class distribution:
Class 0: 2304
Class 1: 768
Class 2: 768
Class 3: 768

Validation set class distribution:
Class 0: 576
Class 1: 192
Class 2: 192
Class 3: 192

Test set 